# 🏛️ Unsloth Studio on Kaggle (Native Cloudflare Tunnel)

> **Old Money Aesthetic Edition** • Automatic Password Capture & Storage in `/tmp`

---

### 📌 Features:
1. **Automatic Password Extraction**: Continuously scans Unsloth Studio console logs for auto-generated passwords, secrets, tokens, and credentials, displaying them prominently in the UI.
2. **Direct Local Port Verification**: Verifies `127.0.0.1:8889` health locally before tunnel creation to eliminate 502 Bad Gateway errors.
3. **60GB+ Scratch Storage Allocation**: Directs ONLY model downloads (`HF_HOME`, `HF_HUB_CACHE`, `TORCH_HOME`, `UNSLOTH_CACHE_DIR`) to Kaggle's `/tmp` scratch space.
4. **Zero Page Snapping**: In-place DOM updates via `display_handle.update()` for smooth scrolling.
5. **Live Diagnostics Console Logs**: Displays real-time stdout/stderr from `[STUDIO]`, `[TUNNEL]`, and `[SYSTEM]`.
6. **Default Password**: unsloth123.

---

> ⚠️ **IMPORTANT**: Ensure **Internet Access** is enabled in your Kaggle Notebook settings (Right Sidebar -> **Settings** -> **Internet: On**).

In [ ]:
# Unsloth Studio on Kaggle

import html
import os
import queue
import re
import shutil
import subprocess
import sys
import textwrap
import threading
import time
import urllib.error
import urllib.request
from IPython.display import HTML, display

STUDIO_PORT = 8889
DEFAULT_PASSWORD = "unsloth123"

# Keep the launcher usable even when Kaggle's PATH does not include user bins.
extra_paths = [os.path.expanduser("~/.local/bin"), "/root/.local/bin", "/usr/local/bin"]
env_path = os.environ.get("PATH", "")
os.environ["PATH"] = ":".join(extra_paths) + ":" + env_path if env_path else ":".join(extra_paths)

# Redirect model downloads and caches to Kaggle's larger /tmp scratch disk.
os.environ["HF_HOME"] = "/tmp/huggingface"
os.environ["HF_HUB_CACHE"] = "/tmp/huggingface/hub"
os.environ["TORCH_HOME"] = "/tmp/torch"
os.environ["UNSLOTH_CACHE_DIR"] = "/tmp/unsloth"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton"

for path in ("/tmp/huggingface", "/tmp/huggingface/hub", "/tmp/torch", "/tmp/unsloth", "/tmp/triton"):
    os.makedirs(path, exist_ok=True)

ANSI = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
LOGS = []


def clean(value):
    return ANSI.sub("", str(value)).replace("\r", "").rstrip()


def esc(value):
    return html.escape(str(value) if value is not None else "")


def local_server_ready(port=STUDIO_PORT):
    try:
        request = urllib.request.Request(
            "http://127.0.0.1:%d" % port,
            headers={"User-Agent": "unsloth-kaggle-launcher/1.0"},
        )
        with urllib.request.build_opener(urllib.request.ProxyHandler({})).open(request, timeout=2):
            return True
    except urllib.error.HTTPError:
        return True
    except Exception:
        return False


CSS = """
<style>
.us-wrap{width:100%;max-width:100%;box-sizing:border-box;overflow:hidden;background:#141210;color:#e8dfd0;border:1px solid #3f382c;padding:26px 30px 22px;font-family:Palatino,"Palatino Linotype",Georgia,serif;}
.us-top{display:flex;justify-content:space-between;align-items:baseline;gap:16px;min-width:0;}
.us-kicker,.us-status,.us-label{font-family:"Courier New",monospace;font-size:10px;letter-spacing:.22em;color:#b8956c;text-transform:uppercase;}
.us-status{text-align:right;overflow-wrap:anywhere;}
.us-title{font-size:27px;color:#f3eadb;margin:7px 0 0;line-height:1.15;overflow-wrap:anywhere;}
.us-rule{height:1px;background:#3f382c;margin:16px 0 18px;border:0;}
.us-sub{color:#a89880;font-size:15px;line-height:1.5;overflow-wrap:anywhere;}
.us-WORKING{color:#d2a05a;}.us-LIVE{color:#9fb58e;}.us-READY{color:#9fb58e;}.us-ERROR{color:#e18b7d;}.us-STOPPED{color:#a89880;}
.us-url{margin:18px 0;padding:17px 18px;background:#0e0c0a;border:1px solid #5a4e3c;text-align:center;overflow:hidden;}
.us-url a{font-family:"Courier New",monospace;font-size:17px;color:#f3eadb;text-decoration:none;border-bottom:1px solid #5a4e3c;word-break:break-all;overflow-wrap:anywhere;}
.us-empty{margin:18px 0;padding:15px;border:1px dashed #5a4e3c;color:#a89880;text-align:center;overflow-wrap:anywhere;}
.us-key{margin:16px 0;padding:14px 16px;background:#0e0c0a;border:1px solid #5a4e3c;overflow:hidden;}
.us-key code{display:block;margin-top:7px;font-family:"Courier New",monospace;font-size:14px;color:#f3eadb;word-break:break-all;overflow-wrap:anywhere;user-select:text;}
.us-row{display:flex;justify-content:space-between;gap:18px;padding:5px 0;font-size:15px;line-height:1.4;min-width:0;}
.us-key-label{color:#8a8070;min-width:150px;flex:0 1 auto;}
.us-value{color:#e8dfd0;text-align:right;word-break:break-word;overflow-wrap:anywhere;min-width:0;}
.us-console{margin-top:16px;border:1px solid #3f382c;background:#0e0c0a;overflow:hidden;}
.us-console pre{margin:0;padding:12px 14px;font-family:"Courier New",monospace;font-size:11px;line-height:1.45;color:#bdb3a0;white-space:pre-wrap;word-break:break-word;overflow-wrap:anywhere;max-height:260px;overflow:auto;}
.us-foot{margin-top:15px;font-family:"Courier New",monospace;font-size:11px;color:#8a8070;line-height:1.5;overflow-wrap:anywhere;}
</style>
"""


class Card:
    def __init__(self):
        self.status = "WORKING"
        self.sub = "Preparing model storage and the Unsloth Studio launcher."
        self.rows = []
        self.foot = "Model caches are redirected to /tmp. Keep the private link and password to yourself."
        self.url = ""
        self.password = ""
        self.handle = None
        self.last_paint = 0.0
        self.paint()

    def paint(self):
        if self.url:
            address = (
                '<div class="us-url"><div class="us-label">Private Cloudflare link</div>'
                '<a href="%s" target="_blank" rel="noopener noreferrer">%s</a></div>'
                % (esc(self.url), esc(self.url))
            )
        else:
            address = '<div class="us-empty">The private Cloudflare link is not connected yet.</div>'

        password = ""
        if self.password:
            password = (
                '<div class="us-key"><div class="us-label">Studio password · sign-in only</div>'
                '<code>%s</code></div>' % esc(self.password)
            )

        rows = "".join(
            '<div class="us-row"><span class="us-key-label">%s</span><span class="us-value">%s</span></div>'
            % (esc(name), esc(value)) for name, value in self.rows[-10:]
        )
        console = "".join(esc(line) + "\n" for line in LOGS[-18:])
        output = HTML(CSS + """
        <div class="us-wrap">
          <div class="us-top"><div class="us-kicker">01 / Launch Studio</div><div class="us-status us-%s">%s</div></div>
          <div class="us-title">Unsloth Studio on Kaggle</div><hr class="us-rule">
          <div class="us-sub">%s</div>%s%s
          <div style="margin-top:14px">%s</div>
          <div class="us-console"><pre>%s</pre></div>
          <div class="us-foot">%s</div>
        </div>""" % (
            esc(self.status), esc(self.status), esc(self.sub), address, password,
            rows, console, esc(self.foot),
        ))
        if self.handle is None:
            self.handle = display(output, display_id="unsloth-studio")
        else:
            self.handle.update(output)
        self.last_paint = time.monotonic()

    def set(self, status=None, sub=None, rows=None, foot=None, url=None, password=None):
        if status is not None:
            self.status = status
        if sub is not None:
            self.sub = sub
        if rows is not None:
            self.rows = rows
        if foot is not None:
            self.foot = foot
        if url is not None:
            self.url = url
        if password is not None:
            self.password = password
        self.paint()

    def log(self, tag, message):
        line = clean(message)
        if not line:
            return
        LOGS.append(textwrap.shorten(
            "%s [%s] %s" % (time.strftime("%H:%M:%S"), tag, line), width=1200,
        ))
        del LOGS[:-200]
        if time.monotonic() - self.last_paint >= 0.15:
            self.paint()


ui = Card()
log_queue = queue.Queue()
studio_proc = None
tunnel_proc = None


def stream_reader(pipe, tag):
    try:
        for line in iter(pipe.readline, ""):
            if line:
                log_queue.put((tag, line))
    except Exception:
        pass
    finally:
        try:
            pipe.close()
        except Exception:
            pass


def drain_logs():
    messages = []
    while True:
        try:
            messages.append(log_queue.get_nowait())
        except queue.Empty:
            break
    return messages


def stop_process(process):
    if process is None:
        return
    try:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
    except Exception:
        pass


def run():
    global studio_proc, tunnel_proc

    base_rows = [
        ("port", str(STUDIO_PORT)),
        ("model cache", "/tmp/huggingface"),
        ("scratch disk", "/tmp (Kaggle)"),
    ]

    ui.set(rows=base_rows + [("studio", "preparing"), ("tunnel", "waiting")])
    ui.log("SYSTEM", "Redirected Hugging Face and PyTorch caches to /tmp.")

    cf_bin = shutil.which("cloudflared")
    if cf_bin:
        ui.log("SYSTEM", "Using Cloudflare binary at %s." % cf_bin)
    else:
        cf_bin = "/tmp/cloudflared"
        if not os.path.exists(cf_bin):
            ui.set(sub="Downloading the Cloudflare Tunnel binary.")
            ui.log("SYSTEM", "Downloading cloudflared to /tmp/cloudflared.")
            try:
                urllib.request.urlretrieve(
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    cf_bin,
                )
                os.chmod(cf_bin, 0o755)
                ui.log("SYSTEM", "Cloudflare Tunnel binary is ready.")
            except Exception as error:
                raise RuntimeError("Could not download cloudflared: %s" % clean(error)) from error
        else:
            ui.log("SYSTEM", "Using cached Cloudflare binary at %s." % cf_bin)

    ui.set(sub="Installing Unsloth Studio and its dependencies.")
    ui.log("SYSTEM", "Installing the Unsloth Python package.")
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location", "unsloth"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False,
        )
        ui.log("SYSTEM", "Python package install finished with code %s." % result.returncode)
        result = subprocess.run(
            "curl -fsSL https://unsloth.ai/install.sh | sh",
            shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False,
        )
        ui.log("SYSTEM", "Unsloth installer finished with code %s." % result.returncode)
    except Exception as error:
        raise RuntimeError("Unsloth installation failed: %s" % clean(error)) from error

    candidates = [
        shutil.which("unsloth"),
        os.path.expanduser("~/.local/bin/unsloth"),
        "/root/.local/bin/unsloth",
        "/usr/local/bin/unsloth",
        "/opt/conda/bin/unsloth",
    ]
    unsloth_bin = next((path for path in candidates if path and os.path.exists(path)), None)
    if unsloth_bin:
        studio_cmd = [unsloth_bin, "studio", "-H", "127.0.0.1", "-p", str(STUDIO_PORT)]
    else:
        studio_cmd = [sys.executable, "-m", "unsloth", "studio", "-H", "127.0.0.1", "-p", str(STUDIO_PORT)]

    ui.set(
        sub="Starting the local Studio server on port %d." % STUDIO_PORT,
        rows=base_rows + [("studio", "starting"), ("tunnel", "waiting")],
    )
    ui.log("SYSTEM", "Launching: %s" % " ".join(studio_cmd))
    studio_proc = subprocess.Popen(
        studio_cmd, env=os.environ,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    threading.Thread(target=stream_reader, args=(studio_proc.stdout, "STUDIO"), daemon=True).start()

    ui.set(sub="Waiting for the local Studio server to answer.")
    server_ready = False
    for attempt in range(30):
        for tag, line in drain_logs():
            ui.log(tag, line)
        if studio_proc.poll() is not None:
            raise RuntimeError(
                "Unsloth Studio exited before its server became ready (code %s)." % studio_proc.returncode
            )
        if local_server_ready():
            server_ready = True
            break
        if attempt and attempt % 5 == 0:
            ui.set(sub="Waiting for the local Studio server on port %d." % STUDIO_PORT)
        time.sleep(1.0)

    if not server_ready:
        raise RuntimeError("Unsloth Studio did not answer on 127.0.0.1:%d." % STUDIO_PORT)

    ui.log("SYSTEM", "Local Studio server is healthy on port %d." % STUDIO_PORT)
    ui.set(
        sub="Opening a private Cloudflare Quick Tunnel.",
        rows=base_rows + [("studio", "online"), ("tunnel", "connecting")],
    )

    tunnel_cmd = [cf_bin, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:%d" % STUDIO_PORT]
    ui.log("SYSTEM", "Launching: %s" % " ".join(tunnel_cmd))
    tunnel_proc = subprocess.Popen(
        tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    threading.Thread(target=stream_reader, args=(tunnel_proc.stdout, "TUNNEL"), daemon=True).start()

    tunnel_url = ""
    captured_pass = ""
    displayed_url = ""
    displayed_pass = ""
    last_wait_update = time.monotonic()

    while True:
        for tag, line in drain_logs():
            text = clean(line)
            if text:
                if not tunnel_url:
                    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
                    if match and "precheck" not in text and "run_id=" not in text and "target=" not in text:
                        tunnel_url = match.group(0)
                        ui.log("SYSTEM", "Cloudflare link captured.")
                if not captured_pass:
                    match = re.search(
                        r"(?:password|token|key|secret)[:\s=]+([a-zA-Z0-9_-]{4,64})",
                        text, re.IGNORECASE,
                    )
                    if match:
                        candidate = match.group(1)
                        if candidate.lower() not in ("true", "false", "none", "null", "info", "error", "warn"):
                            captured_pass = candidate
                            ui.log("SYSTEM", "Studio password captured.")
                ui.log(tag, text)

        if studio_proc.poll() is not None:
            raise RuntimeError("Unsloth Studio stopped unexpectedly (code %s)." % studio_proc.returncode)
        if tunnel_proc.poll() is not None:
            raise RuntimeError("Cloudflare Tunnel stopped unexpectedly (code %s)." % tunnel_proc.returncode)

        if tunnel_url and (tunnel_url != displayed_url or captured_pass != displayed_pass):
            ui.set(
                status="LIVE",
                sub="Unsloth Studio is live. Open the private link and sign in.",
                url=tunnel_url,
                password=captured_pass,
                rows=base_rows + [("studio", "online"), ("tunnel", "connected")],
                foot=(
                    "Use the captured password. If none appears, try %s. "
                    "The link and password are session-scoped; keep both private."
                ) % DEFAULT_PASSWORD,
            )
            displayed_url = tunnel_url
            displayed_pass = captured_pass
            last_wait_update = time.monotonic()
        elif not tunnel_url and time.monotonic() - last_wait_update > 8:
            ui.set(sub="Waiting for Cloudflare to publish the private link.")
            last_wait_update = time.monotonic()

        time.sleep(0.3)


def launch():
    try:
        run()
    except KeyboardInterrupt:
        ui.log("SYSTEM", "Interrupted. Stopping Studio and the tunnel.")
        stop_process(tunnel_proc)
        stop_process(studio_proc)
        ui.set(
            status="STOPPED",
            sub="Unsloth Studio and its tunnel were stopped.",
            url="",
            password="",
            foot="Rerun the cell to start a new session. The link and password will change.",
        )
    except Exception as error:
        ui.log("ERROR", clean(error) or error.__class__.__name__)
        stop_process(tunnel_proc)
        stop_process(studio_proc)
        ui.set(
            status="ERROR",
            sub=clean(error) or error.__class__.__name__,
            url="",
            password="",
            foot="Fix the issue above, then rerun the cell. No model was downloaded by this cell.",
        )


if __name__ == "__main__":
    launch()
